# Analyse Exploratoire des Indices Synthétiques Deriv

Ce notebook contient l'analyse exploratoire des données des indices synthétiques.

## Objectifs:
1. Charger et visualiser les données
2. Analyser les distributions
3. Détecter les patterns (spikes, cycles, etc.)
4. Tester la stationnarité
5. Analyser les autocorrélations
6. Calculer l'exposant de Hurst
7. Analyser les fréquences (FFT)

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.helpers import load_data, calculate_technical_indicators
from src.analyzers.statistical_analyzer import StatisticalAnalyzer
from src.analyzers.pattern_detector import PatternDetector

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement des Données

In [ ]:
# Charger les données
# Choisir l'indice à analyser
SYMBOL = "Volatility_10_Index"  # Modifier selon vos besoins
TIMEFRAME = "M1"

print(f"Loading {SYMBOL} data...")
df = load_data(SYMBOL, TIMEFRAME)

if df is not None:
    print(f"\nData loaded: {len(df)} rows")
    print(f"Period: {df.index[0]} to {df.index[-1]}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst rows:")
    display(df.head())
    print(f"\nBasic statistics:")
    display(df.describe())
else:
    print("Error loading data!")

## 2. Visualisation des Prix

In [ ]:
# Plot interactif avec Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue', width=1)
))

fig.update_layout(
    title=f'{SYMBOL} - Close Price',
    xaxis_title='Time',
    yaxis_title='Price',
    hovermode='x unified',
    height=600
)

fig.show()

In [ ]:
# Calculer les rendements
returns = df['Close'].pct_change().dropna()

# Plot des rendements
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Price', 'Returns'),
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(x=df.index, y=df['Close'], name='Price'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=returns.index, y=returns, name='Returns', line=dict(color='red')),
    row=2, col=1
)

fig.update_layout(height=800, showlegend=True)
fig.show()

## 3. Analyse Statistique Complète

In [ ]:
# Créer l'analyseur
analyzer = StatisticalAnalyzer(df, price_column='Close')

# Analyse complète
print("Running full statistical analysis...")
results = analyzer.full_analysis()

print("\nAnalysis complete!")

### 3.1 Distribution des Rendements

In [ ]:
# Afficher les statistiques de distribution
dist = results['distribution']

print("=== DISTRIBUTION STATISTICS ===")
print(f"Mean:       {dist['mean']:.6f}")
print(f"Std:        {dist['std']:.6f}")
print(f"Skewness:   {dist['skewness']:.4f}")
print(f"Kurtosis:   {dist['kurtosis']:.4f}")
print(f"Min:        {dist['min']:.6f}")
print(f"Max:        {dist['max']:.6f}")

print("\n=== NORMALITY TESTS ===")
for test_name, test_result in dist['normality'].items():
    if isinstance(test_result, dict) and 'is_normal' in test_result:
        print(f"{test_name}: {'Normal' if test_result['is_normal'] else 'NOT Normal'} (p={test_result.get('p_value', 'N/A')})")

In [ ]:
# Visualiser la distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(returns, bins=100, alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Returns')
axes[0].set_xlabel('Returns')
axes[0].set_ylabel('Frequency')
axes[0].axvline(returns.mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

# Q-Q plot
from scipy import stats as sp_stats
sp_stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

### 3.2 Test de Stationnarité

In [ ]:
# Afficher les résultats de stationnarité
stat = results['stationarity']

print("=== STATIONARITY TESTS ===")
print(f"\nADF Test:")
print(f"  Statistic:   {stat['adf']['statistic']:.4f}")
print(f"  P-value:     {stat['adf']['p_value']:.4f}")
print(f"  Result:      {'STATIONARY' if stat['adf']['is_stationary'] else 'NON-STATIONARY'}")

print(f"\nKPSS Test:")
print(f"  Statistic:   {stat['kpss']['statistic']:.4f}")
print(f"  P-value:     {stat['kpss']['p_value']:.4f}")
print(f"  Result:      {'STATIONARY' if stat['kpss']['is_stationary'] else 'NON-STATIONARY'}")

print(f"\nPhillips-Perron Test:")
print(f"  Statistic:   {stat['phillips_perron']['statistic']:.4f}")
print(f"  P-value:     {stat['phillips_perron']['p_value']:.4f}")
print(f"  Result:      {'STATIONARY' if stat['phillips_perron']['is_stationary'] else 'NON-STATIONARY'}")

### 3.3 Autocorrélation

In [ ]:
# Afficher les résultats d'autocorrélation
autocorr = results['autocorrelation']

print("=== AUTOCORRELATION ===")
print(f"Has significant autocorrelation: {autocorr['ljung_box']['has_autocorrelation']}")
print(f"Number of significant lags: {len(autocorr['significant_lags'])}")
if autocorr['significant_lags']:
    print(f"Significant lags: {autocorr['significant_lags'][:10]}...")

In [ ]:
# Plot ACF et PACF
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# ACF
acf_values = autocorr['acf']
axes[0].bar(range(len(acf_values)), acf_values, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=1.96/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=1)
axes[0].axhline(y=-1.96/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=1)
axes[0].set_title('Autocorrelation Function (ACF)')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('ACF')

# PACF
pacf_values = autocorr['pacf']
axes[1].bar(range(len(pacf_values)), pacf_values, alpha=0.7, color='orange')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].axhline(y=1.96/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=1)
axes[1].axhline(y=-1.96/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=1)
axes[1].set_title('Partial Autocorrelation Function (PACF)')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('PACF')

plt.tight_layout()
plt.show()

### 3.4 Exposant de Hurst

In [ ]:
# Afficher l'exposant de Hurst
hurst = results['hurst_exponent']

print("=== HURST EXPONENT ===")
print(f"Hurst Exponent: {hurst['hurst_exponent']:.4f}")
print(f"Interpretation: {hurst['interpretation']}")

if hurst['hurst_exponent'] < 0.5:
    print("\n⚠️  The series is MEAN REVERTING - good for mean reversion strategies!")
elif hurst['hurst_exponent'] > 0.5:
    print("\n📈 The series is TRENDING - good for trend following strategies!")
else:
    print("\n🎲 The series is RANDOM WALK - no predictable pattern")

### 3.5 Analyse Fréquentielle (FFT)

In [ ]:
# Afficher les cycles dominants
freq = results['frequency']

print("=== DOMINANT CYCLES (FFT) ===")
if freq['dominant_periods']:
    for i, (period, power) in enumerate(zip(freq['dominant_periods'][:5], freq['dominant_powers'][:5])):
        print(f"{i+1}. Period: {period:.1f} bars, Power: {power:.2e}")
else:
    print("No dominant cycles detected")

In [ ]:
# Plot du spectre de puissance
frequencies = np.array(freq['frequencies'])
power_spectrum = np.array(freq['power_spectrum'])

# Limiter à des fréquences raisonnables
max_period = 500  # bars
min_freq = 1 / max_period
mask = frequencies >= min_freq

plt.figure(figsize=(15, 5))
plt.plot(1/frequencies[mask], power_spectrum[mask])
plt.xlabel('Period (bars)')
plt.ylabel('Power')
plt.title('Power Spectrum (FFT)')
plt.xscale('log')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

### 3.6 Entropie

In [ ]:
# Afficher les mesures d'entropie
entropy = results['entropy']

print("=== ENTROPY MEASURES ===")
print(f"Shannon Entropy:      {entropy['shannon_entropy']:.4f}")
print(f"Sample Entropy:       {entropy['sample_entropy']:.4f}")
print(f"Approximate Entropy:  {entropy['approximate_entropy']:.4f}")

print("\n💡 Lower entropy = more predictable patterns")
print("💡 Higher entropy = more random behavior")

## 4. Détection de Patterns

In [ ]:
# Créer le détecteur de patterns
detector = PatternDetector(df, price_column='Close')

# Analyse complète
print("Running pattern detection...")
pattern_results = detector.full_pattern_analysis(spike_threshold=3.0)

print("\nPattern detection complete!")

### 4.1 Spikes (Crash/Boom)

In [ ]:
# Afficher les résultats des spikes
spikes = pattern_results['spikes']

print("=== SPIKE DETECTION ===")
print(f"\nCrash spikes detected: {spikes['crash_spikes']['count']}")
if spikes['crash_spikes']['count'] > 0:
    timing = spikes['crash_spikes']['timing']
    print(f"  Mean interval: {timing['mean_interval']:.1f} bars")
    print(f"  Std interval:  {timing['std_interval']:.1f} bars")
    print(f"  Min interval:  {timing['min_interval']} bars")
    print(f"  Max interval:  {timing['max_interval']} bars")

print(f"\nBoom spikes detected: {spikes['boom_spikes']['count']}")
if spikes['boom_spikes']['count'] > 0:
    timing = spikes['boom_spikes']['timing']
    print(f"  Mean interval: {timing['mean_interval']:.1f} bars")
    print(f"  Std interval:  {timing['std_interval']:.1f} bars")
    print(f"  Min interval:  {timing['min_interval']} bars")
    print(f"  Max interval:  {timing['max_interval']} bars")

In [ ]:
# Visualiser les spikes
fig = go.Figure()

# Prix
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Close'],
    mode='lines',
    name='Price',
    line=dict(color='blue', width=1)
))

# Crash spikes
if spikes['crash_spikes']['count'] > 0:
    crash_times = [df.index[i+1] for i in spikes['crash_spikes']['indices']]
    crash_prices = [df['Close'].iloc[i+1] for i in spikes['crash_spikes']['indices']]
    
    fig.add_trace(go.Scatter(
        x=crash_times,
        y=crash_prices,
        mode='markers',
        name='Crash Spikes',
        marker=dict(color='red', size=10, symbol='triangle-down')
    ))

# Boom spikes
if spikes['boom_spikes']['count'] > 0:
    boom_times = [df.index[i+1] for i in spikes['boom_spikes']['indices']]
    boom_prices = [df['Close'].iloc[i+1] for i in spikes['boom_spikes']['indices']]
    
    fig.add_trace(go.Scatter(
        x=boom_times,
        y=boom_prices,
        mode='markers',
        name='Boom Spikes',
        marker=dict(color='green', size=10, symbol='triangle-up')
    ))

fig.update_layout(
    title='Spike Detection',
    xaxis_title='Time',
    yaxis_title='Price',
    height=600,
    hovermode='x unified'
)

fig.show()

### 4.2 Anomalies

In [ ]:
# Afficher les anomalies
anomalies = pattern_results['anomalies']

print("=== ANOMALY DETECTION ===")
print(f"Method:           {anomalies['method']}")
print(f"Anomalies found:  {anomalies['num_anomalies']}")
print(f"Anomaly ratio:    {anomalies['anomaly_ratio']*100:.2f}%")

### 4.3 Cycles Détectés

In [ ]:
# Afficher les cycles
if 'cycles' in pattern_results:
    cycles = pattern_results['cycles']
    
    print("=== DETECTED CYCLES ===")
    print(f"Number of cycles detected: {cycles['num_cycles']}")
    
    if cycles['dominant_cycle']:
        print(f"\nDominant cycle:")
        print(f"  Period:     {cycles['dominant_cycle']['period']:.1f} bars")
        print(f"  Power:      {cycles['dominant_cycle']['power']:.2e}")
        print(f"  Frequency:  {cycles['dominant_cycle']['frequency']:.6f}")
    
    if cycles['detected_cycles']:
        print(f"\nTop 5 cycles:")
        for i, cycle in enumerate(cycles['detected_cycles'][:5]):
            print(f"  {i+1}. Period: {cycle['period']:.1f} bars")

## 5. Conclusion

Résumé des findings:

In [ ]:
print("="*60)
print("ANALYSE SUMMARY")
print("="*60)

print(f"\n📊 SYMBOL: {SYMBOL}")
print(f"📅 Period: {df.index[0]} to {df.index[-1]}")
print(f"📈 Total bars: {len(df)}")

print(f"\n🎲 RANDOMNESS:")
print(f"   Hurst Exponent: {hurst['hurst_exponent']:.4f} ({hurst['interpretation']})")
print(f"   Shannon Entropy: {entropy['shannon_entropy']:.4f}")

print(f"\n📉 DISTRIBUTION:")
print(f"   Mean return: {dist['mean']:.6f}")
print(f"   Std return: {dist['std']:.6f}")
print(f"   Skewness: {dist['skewness']:.4f}")
print(f"   Kurtosis: {dist['kurtosis']:.4f}")

print(f"\n🔄 STATIONARITY:")
print(f"   ADF: {'Stationary' if stat['adf']['is_stationary'] else 'Non-stationary'}")
print(f"   KPSS: {'Stationary' if stat['kpss']['is_stationary'] else 'Non-stationary'}")

print(f"\n⚡ PATTERNS:")
print(f"   Crash spikes: {spikes['crash_spikes']['count']}")
print(f"   Boom spikes: {spikes['boom_spikes']['count']}")
print(f"   Anomalies: {anomalies['num_anomalies']}")
if 'cycles' in pattern_results and pattern_results['cycles']['dominant_cycle']:
    print(f"   Dominant cycle: {pattern_results['cycles']['dominant_cycle']['period']:.1f} bars")

print("\n" + "="*60)

## 6. Prochaines Étapes

Basé sur cette analyse, les prochaines étapes pourraient être:

1. **Si mean-reverting (H < 0.5)**: Développer des stratégies de mean reversion
2. **Si trending (H > 0.5)**: Développer des stratégies de trend following
3. **Si spikes détectés**: Modéliser la probabilité et le timing des spikes
4. **Si cycles détectés**: Exploiter les cycles pour le timing d'entrée/sortie
5. **Machine Learning**: Utiliser ces features pour entraîner des modèles prédictifs